In [1]:
# Upgrade PyTorch to a unified CUDA 12.1 version
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install and upgrade all Hugging Face and evaluation libraries together
!pip install --upgrade transformers datasets evaluate accelerate peft bitsandbytes sentencepiece rouge_score bert_score nltk

Looking in indexes: https://download.pytorch.org/whl/cu121
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 80.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 91.8 MB/s eta 0:00:00:00:01
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully un

In [2]:
# !pip install --upgrade torchao

In [3]:
import gc
import torch
import os
import json
import glob
import numpy as np
import evaluate
from datasets import Dataset
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [4]:
def migrate_essay_data(folder_path):
    all_essay_records = []
    search_pattern = os.path.join(folder_path, "*_essay.jsonl")
    essay_files = glob.glob(search_pattern)
    for file_path in essay_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        record = json.loads(line)
                        if record.get("type") == "essay" and record.get("model_answer"):
                            all_essay_records.append(record)
                    except: continue
    return all_essay_records

In [5]:
def preprocess_function(examples):
    inputs = [f"summarize technical info: {c}" for c in examples["context"]]
    model_inputs = tokenizer(inputs, max_length=MAX_SOURCE_LENGTH, truncation=True)

    targets = [f"Scenario: {s} Question: {q} Answer: {a}" 
               for s, q, a in zip(examples["scenario_context"], examples["question"], examples["model_answer"])]
    
    labels = tokenizer(text_target=targets, max_length=MAX_TARGET_LENGTH, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs 

In [6]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)

In [7]:
MODEL_NAME = "google/flan-t5-large"   
DATA_FOLDER = "/kaggle/input/datasets/adhamashraf202200953/technical-parsed-questions"
OUTPUT_DIR = "./outputs/STABLE_T5_LARGE" 

BATCH_SIZE = 2             
GRADIENT_ACC_STEPS = 16    
MAX_SOURCE_LENGTH = 384
MAX_TARGET_LENGTH = 256

gc.collect()
torch.cuda.empty_cache()

raw_essay_data = migrate_essay_data(DATA_FOLDER)
dataset = Dataset.from_list(raw_essay_data)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
split_dataset = tokenized_dataset.train_test_split(test_size=0.1)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, 
    device_map="auto",
    torch_dtype=torch.float32 
)

peft_config = LoraConfig(
    r=16, 
    lora_alpha=32, 
    target_modules=["q", "v", "wi_0", "wi_1", "wo"], 
    lora_dropout=0.1, 
    task_type=TaskType.SEQ_2_SEQ_LM
)
model = get_peft_model(model, peft_config)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [8]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",
    eval_steps=50,               
    logging_steps=5,
    learning_rate=2e-4,         
    num_train_epochs=3,          
    max_grad_norm=1.0,           
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACC_STEPS,
    fp16=False,                  
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    optim="adamw_torch",         
    report_to="none"
)

In [11]:
rouge_metric = evaluate.load("rouge")

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, 
        model=model, 
        padding=True,
        label_pad_token_id=-100  
    ),
    compute_metrics=compute_metrics
)

trainer.train()

Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
50,37.607657,2.082935,0.323196,0.077258,0.201481,0.201268
100,34.098077,1.947044,0.363865,0.100042,0.230431,0.230000
150,33.465656,1.903519,0.370130,0.105845,0.235965,0.235566
189,32.973923,1.896246,0.367491,0.103482,0.234659,0.234291


TrainOutput(global_step=189, training_loss=35.79762954308242, metrics={'train_runtime': 8378.1049, 'train_samples_per_second': 0.712, 'train_steps_per_second': 0.023, 'total_flos': 1.0186502909386752e+16, 'train_loss': 35.79762954308242, 'epoch': 3.0})